In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from langchain_google_genai import ChatGoogleGenerativeAI
import faiss
import spacy
import contractions
from textblob import TextBlob

c:\Users\sushm\AppData\Local\Programs\Python\Python314\Lib\site-packages\langchain_core\utils\pydantic.py:41: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


### load the document

In [2]:
data=open("data.txt").read()

In [3]:
data = data.lower()

### Removing extra space

In [4]:
import re
data=re.sub(r'\s{2,}','',data )

removing numbers like 1 ,2,3

In [5]:
#data=re.sub(r'')

### contractions

In [6]:
contractions.fix(data)


'machine learning is the science of teaching computers to learn from data.\nit is a central branch of artificial intelligence.\nunlike traditional programming, ml does not rely on explicit rules.\ninstead, it infers rules from examples.\nthe essence of ml lies in generalization.\na model trained on past data must perform well on unseen data.\nthis ability makes ml powerful and versatile.\nml begins with data collection.\ndata can be structured, semi-structured, or unstructured.\nstructured data includes tables in sql.\nsemi-structured data includes logs or json files.\nunstructured data includes images, audio, and text.\nthe richness of data determines model capability.\nalgorithms process this data to uncover relationships.\nlinear regression models simple numeric relationships.\ndecision trees split data into hierarchical rules.\nsupport vector machines find optimal boundaries.\nneural networks mimic the human brain.\ndeep learning uses multiple layers of neural networks.\nit powers 

4.Removing punctuation and special characters

In [7]:
data=re.sub(r'[^0-9a-zA-Z\s]','',data)

### Textblob

In [8]:
values=TextBlob(data).correct()
values

TextBlob("machine learning is the science of teaching computers to learn from data
it is a central branch of artificial intelligence
unlike traditional programming my does not rely on explicit rules
instead it infer rules from examples
the essence of my lies in generalization
a model trained on past data must perform well on unseen data
this ability makes my powerful and versatile
my begins with data collection
data can be structures semistructured or unstructured
structures data includes tables in sal
semistructured data includes logs or son files
unstructured data includes images audit and text
the richness of data determines model capability
algorithms process this data to uncover relationships
linear repression models simple numerical relationships
decision trees split data into hierarchical rules
support vector machines find optical boundaries
neutral network mimi the human brain
deep learning uses multiple layers of neutral network
it powers breakthroughs in vision and language
m

### spacy

### lemmatization

In [9]:
import spacy
nlp = spacy.load('en_core_web_sm')
tokens=nlp(data)
updated_tokens=[token.lemma_ for token  in tokens if not token.is_stop]
data = ' '.join(updated_tokens).strip()
data

'machine learning science teach computer learn datum \n central branch artificial intelligence \n unlike traditional programming ml rely explicit rule \n instead infer rule example \n essence ml lie generalization \n model train past datum perform unseen datum \n ability make ml powerful versatile \n ml begin data collection \n datum structure semistructure unstructured \n structured datum include table sql \n semistructure datum include log json file \n unstructured datum include image audio text \n richness datum determine model capability \n algorithm process datum uncover relationship \n linear regression model simple numeric relationship \n decision tree split datum hierarchical rule \n support vector machine find optimal boundary \n neural network mimic human brain \n deep learning use multiple layer neural network \n power breakthrough vision language \n ml divide supervise learning \n divide unsupervised learning \n reinforcement learning form paradigm \n supervise learning use

### chunking(converting docs into chunks)

In [10]:
text_splitter=RecursiveCharacterTextSplitter(
                  chunk_size=200,
                  chunk_overlap=40,
)

chunks=list(set(text_splitter.split_text(data)))

### embeddings(coverts chunks into vectors)

In [11]:
embeddings_model=SentenceTransformer(
     model_name_or_path='sentence-transformers/all-miniLM-L6-V2'
)

chunk_embeddings=embeddings_model.encode(chunks).astype("float32")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [12]:
dimension=chunk_embeddings.shape[1]
dimension

384

In [13]:
faiss.normalize_L2(chunk_embeddings)

In [14]:
index_faiss_db=faiss.IndexFlatIP(dimension)
index_faiss_db.add(chunk_embeddings)

In [ ]:
def rag_query(query,k=2):
    

    query_embeddings=embeddings_model.encode(query).astype("float32")
    query_embeddings=query_embeddings.reshape(1,-1)
    faiss.normalize_L2(query_embeddings)
    distance,index = index_faiss_db.search(query_embeddings,k=k)

    R_chunks=[chunks[i] for i in index[0]]
    R_str=" ".join(R_chunks)
    return R_str

    prompt =f'''
              you are an helpfull assistant
              Assigned task for you: Structure my output => {R_str}
            note:
            1)don't add extra contents just structure mentioned output.
            2)if there is any mistakes in output correct or else keep the original output with structured result'''
    llm_model=ChatGoogleGenerativeAI(
    model="gemini-2.5-flash"   #gemini-3.5-flash
    )

user_prompt='Explain Machine Learning'
user_prompt=re.sub(r'[^0-9a-zA-Z\s]','',user_prompt)



response = rag_query(user_prompt)
print(response)

machine learning create 
 machine learning innovate 
 machine learning discover 
 machine learning imagine 
 machine learning evolve 
 machine learning progress 
 machine learning succeed machine learning unite 
 machine learning guide 
 machine learning shape 
 machine learning define 
 machine learning create 
 machine learning innovate 
 machine learning discover


In [16]:
def r_search(query,k=3):
    query_embeddings = embeddings_model.encode(query).astype('float32')
    query_embeddings = query_embeddings.reshape(1,-1)
    faiss.normalize_L2(query_embeddings)
    print(query_embeddings.shape)
    distance,index = index_faiss_db.search(query_embeddings,k=k)
    R_chunks = [chunks[i] for i in index[0]]
    R_str = ' '.join(R_chunks)
    return R_str
def g_text(r_search):
        import os
        import requests
    
        API_URL = "https://router.huggingface.co/v1/chat/completions"
    
        headers = {
            "Authorization": f"Bearer {os.environ['HF_TOKEN']}",
        }
        def query(payload):
            response = requests.post(API_URL, headers=headers, json=payload)
            return response.json()
        prompt = f'''
                    You're an helpful assistant
                    Assigned Task for you : Structure my output => {r_search}
                    Note : 
                    1) Don't add extra contents just structure mentioned output.
                    2) If there is mistake in output correct or else keep the original output
                    with structured result.
            '''
        response = query({
            "messages": [
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            "model": "deepseek-ai/DeepSeek-R1:novita"
        })
    
        return response
user_prompt = 'Explain Machine Learning ?'
user_prompt = re.sub(r'[^0-9a-zA-Z\s]','',user_prompt)

r_response = r_search(user_prompt)
g_response = g_text(r_response)
print(g_response)

(1, 384)


JSONDecodeError: Expecting value: line 1 column 1 (char 0)